In [ ]:
# zelle 1

In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml


def find_project_root(start: Path) -> Path:
    """
    Suche vom aktuellen Ordner aus nach oben nach einem
    Projektordner, der src/walinet enthält.
    """
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (candidate / "src" / "walinet").is_dir():
            return candidate

    raise FileNotFoundError(
        "WALINET project root not found. "
        "Start the notebook somewhere inside the repository "
        "or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = find_project_root(
    Path.cwd()
)

src_dir = PROJECT_ROOT / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(
        0,
        str(src_dir),
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src directory:", src_dir)

In [ ]:
TRAIN_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "Training"
    / "train_7T.yaml"
)

SIMULATION_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "Simulation"
    / "7T_on_the_fly.yaml"
)

print("Training config:")
print(TRAIN_CONFIG_PATH)
print("Exists:", TRAIN_CONFIG_PATH.is_file())

print()

print("Simulation config:")
print(SIMULATION_CONFIG_PATH)
print("Exists:", SIMULATION_CONFIG_PATH.is_file())

assert TRAIN_CONFIG_PATH.is_file(), (
    "Training config not found:\n"
    f"{TRAIN_CONFIG_PATH}"
)

assert SIMULATION_CONFIG_PATH.is_file(), (
    "Simulation config not found:\n"
    f"{SIMULATION_CONFIG_PATH}"
)

In [ ]:
from walinet.config.build import (
    build_config,
)
from walinet.config.build_simulation import (
    build_simulation_config,
)


def load_yaml(
    path: Path,
) -> dict:
    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        raw = yaml.safe_load(
            file
        )

    if not isinstance(
        raw,
        dict,
    ):
        raise TypeError(
            "Expected a YAML mapping in:\n"
            f"{path}\n"
            f"Found: {type(raw)}"
        )

    return raw


train_raw = load_yaml(
    TRAIN_CONFIG_PATH
)

simulation_raw = load_yaml(
    SIMULATION_CONFIG_PATH
)

train_cfg = build_config(
    train_raw,
    config_dir=TRAIN_CONFIG_PATH.parent,
)

simulation_cfg = (
    build_simulation_config(
        simulation_raw,
        config_dir=(
            SIMULATION_CONFIG_PATH.parent
        ),
    )
)

print(
    "Both configs loaded and "
    "validated successfully."
)

In [ ]:
print("TRAINING CONFIG")
print("=" * 60)

print(
    "Run name:",
    train_cfg.run.name,
)

print(
    "Data source:",
    train_cfg.data.source,
)

print(
    "Base directory:",
    train_cfg.data.base_dir,
)

print(
    "Train subjects:",
    len(
        train_cfg.data.train_subjects
    ),
)

print(
    "Validation subjects:",
    len(
        train_cfg.data.val_subjects
    ),
)

print(
    "Normalization:",
    train_cfg.data.normalization,
)

print(
    "Training batch size:",
    train_cfg.training.batch_size,
)

print(
    "Validation spectra:",
    train_cfg.validation.n_spectra,
)

print(
    "Validation seed:",
    train_cfg.validation.seed,
)


print()
print("SIMULATION CONFIG")
print("=" * 60)

print(
    "Version:",
    simulation_cfg.version,
)

print(
    "Bandwidth [Hz]:",
    simulation_cfg
    .acquisition
    .bandwidth_hz,
)

print(
    "Target timepoints:",
    simulation_cfg
    .acquisition
    .n_timepoints,
)

print(
    "NMR frequency [Hz]:",
    simulation_cfg
    .acquisition
    .nmr_frequency_hz,
)

print(
    "Subject mixing:",
    simulation_cfg
    .subject_sampling
    .mixing,
)

print(
    "Lipid projection enabled:",
    simulation_cfg
    .lipid_projection
    .enabled,
)

print(
    "Basis config:",
    simulation_cfg
    .basis
    .config,
)

print(
    "Metabolite config:",
    simulation_cfg
    .metabolites
    .config,
)

In [ ]:
assert (
    train_cfg.data.source
    == "on_the_fly"
)

assert (
    train_cfg.data.on_the_fly
    is not None
)

resource_template = (
    train_cfg
    .data
    .on_the_fly
    .resources
    .filename
)

resource_version = (
    train_cfg
    .data
    .on_the_fly
    .resources
    .version
)

resource_relative_path = Path(
    resource_template.format(
        version=resource_version,
    )
)

base_dir = Path(
    train_cfg.data.base_dir
)

print(
    "Relative resource path:",
    resource_relative_path,
)

print(
    "Base directory:",
    base_dir,
)

In [ ]:
def collect_resource_paths(
    subjects: list[str],
) -> pd.DataFrame:
    rows = []

    for subject in subjects:
        path = (
            base_dir
            / subject
            / resource_relative_path
        )

        rows.append(
            {
                "subject": subject,
                "resource_path": str(path),
                "exists": path.is_file(),
                "size_mb": (
                    path.stat().st_size
                    / 1024**2
                    if path.is_file()
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


train_paths = collect_resource_paths(
    train_cfg.data.train_subjects
)

validation_paths = (
    collect_resource_paths(
        train_cfg.data.val_subjects
    )
)

print("Training resources")
display(train_paths)

print("Validation resources")
display(validation_paths)


missing_train = train_paths.loc[
    ~train_paths["exists"],
    "subject",
].tolist()

missing_validation = (
    validation_paths.loc[
        ~validation_paths["exists"],
        "subject",
    ].tolist()
)

assert not missing_train, (
    "Missing training resources:\n"
    f"{missing_train}"
)

assert not missing_validation, (
    "Missing validation resources:\n"
    f"{missing_validation}"
)

print(
    "All expected resource files exist."
)

In [ ]:
from walinet.training_data.simulation_resources import (
    SimulationPool,
    SimulationResources,
    build_simulation_resources,
)


resources = build_simulation_resources(
    train_cfg=train_cfg,
    simulation_cfg=simulation_cfg,
)

print(
    "Resource pools created successfully."
)

In [ ]:
def tensor_size_mb(
    tensor: torch.Tensor | None,
) -> float:
    if tensor is None:
        return 0.0

    return (
        tensor.numel()
        * tensor.element_size()
        / 1024**2
    )

In [ ]:
def summarize_pool(
    name: str,
    pool: SimulationPool,
) -> None:
    print(name)
    print("=" * 60)

    print(
        "Subjects:",
        pool.n_subjects,
    )

    print(
        "Subject names:",
        pool.subject_names,
    )

    print(
        "Water FIDs:",
        tuple(
            pool.water_fids.shape
        ),
        pool.water_fids.dtype,
    )

    print(
        "Lipid FIDs:",
        tuple(
            pool.lipid_fids.shape
        ),
        pool.lipid_fids.dtype,
    )

    print(
        "Water offsets:",
        tuple(
            pool.water_offsets.shape
        ),
    )

    print(
        "Lipid offsets:",
        tuple(
            pool.lipid_offsets.shape
        ),
    )

    print(
        "Native lengths:",
        pool.native_lengths.tolist(),
    )

    print(
        "Bandwidth [Hz]:",
        pool.bandwidth_hz,
    )

    print(
        "Target timepoints:",
        pool.n_timepoints,
    )

    print(
        "Device:",
        pool.device,
    )

    print(
        "Water memory [MB]:",
        round(
            tensor_size_mb(
                pool.water_fids
            ),
            2,
        ),
    )

    print(
        "Lipid memory [MB]:",
        round(
            tensor_size_mb(
                pool.lipid_fids
            ),
            2,
        ),
    )

    print(
        "Projection memory [MB]:",
        round(
            tensor_size_mb(
                pool
                .lipid_projection_operators
            ),
            2,
        ),
    )

    total_memory = (
        tensor_size_mb(
            pool.water_fids
        )
        + tensor_size_mb(
            pool.lipid_fids
        )
        + tensor_size_mb(
            pool
            .lipid_projection_operators
        )
    )

    print(
        "Total main tensor memory [MB]:",
        round(
            total_memory,
            2,
        ),
    )

    print()


summarize_pool(
    "TRAIN POOL",
    resources.train,
)

summarize_pool(
    "VALIDATION POOL",
    resources.validation,
)

In [ ]:
def validate_pool(
    pool: SimulationPool,
    expected_subjects: list[str],
) -> None:
    target_t = (
        simulation_cfg
        .acquisition
        .n_timepoints
    )

    expected_bandwidth = (
        simulation_cfg
        .acquisition
        .bandwidth_hz
    )

    assert (
        pool.subject_names
        == tuple(expected_subjects)
    )

    assert (
        pool.n_subjects
        == len(expected_subjects)
    )

    assert pool.water_fids.ndim == 2
    assert pool.lipid_fids.ndim == 2

    assert (
        pool.water_fids.shape[1]
        == target_t
    )

    assert (
        pool.lipid_fids.shape[1]
        == target_t
    )

    assert (
        pool.water_fids.dtype
        == torch.complex64
    )

    assert (
        pool.lipid_fids.dtype
        == torch.complex64
    )

    assert (
        pool.water_offsets.dtype
        == torch.int64
    )

    assert (
        pool.lipid_offsets.dtype
        == torch.int64
    )

    assert (
        pool.native_lengths.dtype
        == torch.int64
    )

    assert (
        pool.water_offsets.shape
        == (pool.n_subjects + 1,)
    )

    assert (
        pool.lipid_offsets.shape
        == (pool.n_subjects + 1,)
    )

    assert (
        pool.native_lengths.shape
        == (pool.n_subjects,)
    )

    assert (
        int(
            pool.water_offsets[0]
        )
        == 0
    )

    assert (
        int(
            pool.lipid_offsets[0]
        )
        == 0
    )

    assert torch.all(
        pool.water_offsets[1:]
        >= pool.water_offsets[:-1]
    )

    assert torch.all(
        pool.lipid_offsets[1:]
        >= pool.lipid_offsets[:-1]
    )

    assert (
        int(
            pool.water_offsets[-1]
        )
        == pool.n_water_fids
    )

    assert (
        int(
            pool.lipid_offsets[-1]
        )
        == pool.n_lipid_fids
    )

    assert torch.all(
        pool.water_counts > 0
    )

    assert torch.all(
        pool.lipid_counts > 0
    )

    assert torch.all(
        pool.native_lengths > 0
    )

    assert torch.isfinite(
        pool.water_fids.real
    ).all()

    assert torch.isfinite(
        pool.water_fids.imag
    ).all()

    assert torch.isfinite(
        pool.lipid_fids.real
    ).all()

    assert torch.isfinite(
        pool.lipid_fids.imag
    ).all()

    assert not torch.any(
        torch.all(
            pool.water_fids == 0,
            dim=-1,
        )
    )

    assert not torch.any(
        torch.all(
            pool.lipid_fids == 0,
            dim=-1,
        )
    )

    assert (
        pool.n_timepoints
        == target_t
    )

    assert np.isclose(
        pool.bandwidth_hz,
        expected_bandwidth,
    )

    if (
        simulation_cfg
        .lipid_projection
        .enabled
    ):
        assert (
            pool
            .lipid_projection_operators
            is not None
        )

        assert (
            pool
            .lipid_projection_operators
            .shape
            == (
                pool.n_subjects,
                target_t,
                target_t,
            )
        )

        assert (
            pool
            .lipid_projection_operators
            .dtype
            == torch.complex64
        )

        assert torch.isfinite(
            pool
            .lipid_projection_operators
            .real
        ).all()

        assert torch.isfinite(
            pool
            .lipid_projection_operators
            .imag
        ).all()

    else:
        assert (
            pool
            .lipid_projection_operators
            is None
        )

In [ ]:
validate_pool(
    resources.train,
    train_cfg.data.train_subjects,
)

validate_pool(
    resources.validation,
    train_cfg.data.val_subjects,
)


overlap = (
    set(
        resources
        .train
        .subject_names
    )
    & set(
        resources
        .validation
        .subject_names
    )
)

assert not overlap, (
    "Train/validation overlap found:\n"
    f"{sorted(overlap)}"
)

print(
    "All structural pool checks passed."
)

In [ ]:
def subject_table(
    pool: SimulationPool,
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "subject_index": (
                np.arange(
                    pool.n_subjects
                )
            ),
            "subject": (
                pool.subject_names
            ),
            "water_fids": (
                pool
                .water_counts
                .cpu()
                .numpy()
            ),
            "lipid_fids": (
                pool
                .lipid_counts
                .cpu()
                .numpy()
            ),
            "native_timepoints": (
                pool
                .native_lengths
                .cpu()
                .numpy()
            ),
        }
    )


print("Training subjects")

display(
    subject_table(
        resources.train
    )
)


print("Validation subjects")

display(
    subject_table(
        resources.validation
    )
)

In [ ]:
POOL_TO_INSPECT = (
    resources.train
)

SUBJECT_INDEX = 0


subject_name = (
    POOL_TO_INSPECT
    .subject_name(
        SUBJECT_INDEX
    )
)

water_subject = (
    POOL_TO_INSPECT
    .water_for_subject(
        SUBJECT_INDEX
    )
)

lipid_subject = (
    POOL_TO_INSPECT
    .lipids_for_subject(
        SUBJECT_INDEX
    )
)


print(
    "Subject:",
    subject_name,
)

print(
    "Water subset:",
    tuple(
        water_subject.shape
    ),
)

print(
    "Lipid subset:",
    tuple(
        lipid_subject.shape
    ),
)

In [ ]:
water_start = int(
    POOL_TO_INSPECT
    .water_offsets[
        SUBJECT_INDEX
    ]
)

water_end = int(
    POOL_TO_INSPECT
    .water_offsets[
        SUBJECT_INDEX + 1
    ]
)

lipid_start = int(
    POOL_TO_INSPECT
    .lipid_offsets[
        SUBJECT_INDEX
    ]
)

lipid_end = int(
    POOL_TO_INSPECT
    .lipid_offsets[
        SUBJECT_INDEX + 1
    ]
)


assert torch.equal(
    water_subject,
    POOL_TO_INSPECT
    .water_fids[
        water_start:water_end
    ],
)

assert torch.equal(
    lipid_subject,
    POOL_TO_INSPECT
    .lipid_fids[
        lipid_start:lipid_end
    ],
)

print(
    "Subject helper methods agree "
    "with the stored offsets."
)

In [ ]:
WATER_EXAMPLE_INDEX = 0

water_fid = (
    water_subject[
        WATER_EXAMPLE_INDEX
    ]
    .cpu()
    .numpy()
)

water_spectrum = np.fft.fftshift(
    np.fft.fft(
        water_fid
    )
)


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        water_fid
    )
)

plt.title(
    f"Water FID – {subject_name}"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.show()


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        water_spectrum
    )
)

plt.title(
    f"Water spectrum – {subject_name}"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Magnitude"
)

plt.show()

In [ ]:
LIPID_EXAMPLE_INDEX = 0

lipid_fid = (
    lipid_subject[
        LIPID_EXAMPLE_INDEX
    ]
    .cpu()
    .numpy()
)

lipid_spectrum = np.fft.fftshift(
    np.fft.fft(
        lipid_fid
    )
)


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        lipid_fid
    )
)

plt.title(
    f"Lipid FID – {subject_name}"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.show()


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        lipid_spectrum
    )
)

plt.title(
    f"Lipid spectrum – {subject_name}"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Magnitude"
)

plt.show()

In [ ]:
def sample_global_indices_for_subjects(
    offsets: torch.Tensor,
    subject_indices: torch.Tensor,
    *,
    generator: torch.Generator,
) -> torch.Tensor:
    """
    Wähle für jedes angegebene Subject einen zufälligen
    lokalen FID-Index und rechne ihn in einen globalen
    Pool-Index um.
    """
    counts = (
        offsets[1:]
        - offsets[:-1]
    )

    selected_counts = counts[
        subject_indices
    ]

    random_values = torch.rand(
        subject_indices.shape,
        generator=generator,
        device=subject_indices.device,
    )

    local_indices = torch.floor(
        random_values
        * selected_counts
    ).to(
        torch.int64
    )

    global_indices = (
        offsets[
            subject_indices
        ]
        + local_indices
    )

    return global_indices

In [ ]:
test_generator_1 = (
    torch.Generator(
        device="cpu"
    )
)

test_generator_1.manual_seed(
    12345
)


test_generator_2 = (
    torch.Generator(
        device="cpu"
    )
)

test_generator_2.manual_seed(
    12345
)


subject_indices = torch.tensor(
    [
        0,
        0,
        1,
        1,
        resources.train.n_subjects - 1,
    ],
    dtype=torch.int64,
)


indices_1 = (
    sample_global_indices_for_subjects(
        resources
        .train
        .water_offsets,
        subject_indices,
        generator=test_generator_1,
    )
)

indices_2 = (
    sample_global_indices_for_subjects(
        resources
        .train
        .water_offsets,
        subject_indices,
        generator=test_generator_2,
    )
)


print(
    "Subject indices:",
    subject_indices.tolist(),
)

print(
    "Sampled global water indices:",
    indices_1.tolist(),
)


assert torch.equal(
    indices_1,
    indices_2,
)


for (
    subject_index,
    global_index,
) in zip(
    subject_indices.tolist(),
    indices_1.tolist(),
):
    start = int(
        resources
        .train
        .water_offsets[
            subject_index
        ]
    )

    end = int(
        resources
        .train
        .water_offsets[
            subject_index + 1
        ]
    )

    assert (
        start
        <= global_index
        < end
    )


print(
    "Deterministic subject-aware "
    "index sampling works."
)

In [ ]:
operators = (
    resources
    .train
    .lipid_projection_operators
)


if operators is None:
    print(
        "No projection operators were loaded."
    )

    print(
        "This is expected when "
        "lipid_projection.enabled is false."
    )

else:
    print(
        "Projection operators:",
        tuple(
            operators.shape
        ),
        operators.dtype,
    )

    operator = (
        operators[
            SUBJECT_INDEX
        ]
        .cpu()
        .numpy()
    )

    print(
        "Selected operator shape:",
        operator.shape,
    )

    print(
        "Finite:",
        np.isfinite(
            operator
        ).all(),
    )